# Tractian GTM Data Engineering Case Study

**Name:** Mary Mullinax

**Input:** Company name + website URL  
**Output:** ICP score + facility list → CSV ready for CRM upload

## Step 1: Import Libraries

In [2]:
!pip install beautifulsoup4

In [3]:
import sys
!{sys.executable} -m pip install beautifulsoup4

import sys
!{sys.executable} -m pip install pandas

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [4]:
import requests                      # visits websites
from bs4 import BeautifulSoup        # reads the text on those pages
import pandas as pd                  # builds the output table

/Users/marymullinax/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Step 2: Define Your Target Companies
Select companies you want to add to the list.
I selected Amcor, Crown Holdings, and Sonoco Products. 

It's important to note that Amcor just merged with Berry Global


In [57]:
COMPANIES = [
    # Tier 1 — Strong fit (packaging / industrial manufacturing)
    {"name": "Amcor",            "website": "https://www.amcor.com"},
    {"name": "Crown Holdings",   "website": "https://www.crowncork.com"},
    {"name": "Sonoco Products",  "website": "https://www.sonoco.com"},
    {"name": "Berry Global",     "website": "https://www.berryglobal.com"},
    {"name": "Sealed Air",       "website": "https://www.sealedair.com"},
    {"name": "Greif",            "website": "https://www.greif.com"},
    {"name": "Silgan Holdings",  "website": "https://www.silgan.com"},
    {"name": "Pactiv Evergreen", "website": "https://www.pactivevergreen.com"},
    {"name": "Ardagh Group",     "website": "https://www.ardaghgroup.com"},
    {"name": "Graphic Packaging","website": "https://www.graphicpkg.com"},
    {"name": "Coca-Cola",        "website": "https://www.coca-cola.com"},
    {"name": "Procter & Gamble", "website": "https://www.pg.com"},

    # Tier 2 — Partial fit (industrial but not packaging)
    {"name": "Komatsu",          "website": "https://www.komatsu.com"},
    {"name": "Caterpillar",      "website": "https://www.caterpillar.com"},
    {"name": "John Deere",       "website": "https://www.deere.com"},
    {"name": "Parker Hannifin",  "website": "https://www.parker.com"},
    {"name": "Emerson Electric", "website": "https://www.emerson.com"},
    {"name": "Nucor",           "website": "https://www.nucor.com"},
    {"name": "LyondellBasell",    "website": "https://www.lyondellbasell.com"},
    {"name": "Eastman",          "website": "https://www.eastman.com"},

# Tier 3 — Poor fit (wrong sector entirely)
    {"name": "Apple",            "website": "https://www.apple.com"},
    {"name": "Spotify",          "website": "https://www.spotify.com"},
    {"name": "Airbnb",           "website": "https://www.airbnb.com"},
]

print(f"{len(COMPANIES)} companies queued")

23 companies queued


## Step 3: ICP Scoring Rules
This is the "ideal customer" definition.  
Change the weights and thresholds to match your actual ICP.

In [58]:
# Each signal adds points to the ICP score (max 10)
# Edit these to match your real ICP criteria

ICP_RULES = {
    "packaging_keywords": {
        "keywords": ["packaging", "container", "plastic", "flexible", "rigid", "film", "label"],
        "points": 3,
        "reason": "Core packaging sector match"
    },
    "global_keywords": {
        "keywords": ["global", "worldwide", "international", "locations", "facilities", "plants"],
        "points": 2,
        "reason": "Multi-site global footprint"
    },
    "industrial_keywords": {
        "keywords": ["manufacturing", "industrial", "production", "operations", "supply chain", "equipment", 
                     "machinery", "agriculture", "heavy equipment", "fleet", "mining", "construction"],
        "points": 2,
        "reason": "Industrial manufacturer"
    },
    "size_keywords": {
        "keywords": ["billion", "Fortune", "NYSE", "NASDAQ", "annual report", "investors"],
        "points": 2,
        "reason": "Large enterprise signals"
    },
    "sustainability_keywords": {
        "keywords": ["sustainability", "recycle", "circular", "ESG", "carbon"],
        "points": 1,
        "reason": "Sustainability focus (bonus)"
    },
}

print("ICP rules defined with the signals listed below.")
print("Signals:", list(ICP_RULES.keys()))

ICP rules defined with the signals listed below.
Signals: ['packaging_keywords', 'global_keywords', 'industrial_keywords', 'size_keywords', 'sustainability_keywords']


## Step 4: Scraper
Visits the company website, tries common pages, returns the text it finds.  


In [59]:
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; research-bot/1.0)"}

# Pages most likely to have location info — tries each one in order
PAGES_TO_TRY = [
    "/locations",
    "/global-locations",
    "/about",
    "/about-us",
    "/contact",
    "/facilities",
    "/manufacturing",
    "/our-locations",
    "/where-we-are",
]

def scrape_company(website):
    """
    Visit a company's website and collect text from key pages.
    Returns a single string of all text found.
    """
    base = website.rstrip("/")
    all_text = []

    for path in PAGES_TO_TRY:
        url = base + path
        try:
            response = requests.get(url, headers=HEADERS, timeout=6)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, "html.parser")
                # Strip nav/footer noise
                for tag in soup(["script", "style", "nav", "footer"]):
                    tag.decompose()
                text = soup.get_text(separator=" ", strip=True)
                if len(text) > 300:           # only keep pages with real content
                    all_text.append(text[:3000])  # cap each page at 3000 chars
        except:
            pass  # if a page fails, skip it silently

    return " ".join(all_text)

print("Scraper ready.")

Scraper ready.


## Step 5 — ICP Scorer
Reads the scraped text, checks which keywords appear, adds up the score.


In [60]:
def score_company(text):
    """
    Check the scraped text against ICP rules.
    Returns a score out of 10 and a list of matched signals.
    """
    text_lower = text.lower()
    total_score = 0
    matched_signals = []

    for signal_name, rule in ICP_RULES.items():
        # Check if ANY keyword from this signal appears in the text
        if any(kw in text_lower for kw in rule["keywords"]):
            total_score += rule["points"]
            matched_signals.append(rule["reason"])

    # Cap score at 10
    final_score = min(total_score, 10)

    # Assign tier
    if final_score >= 8:
        tier = "Tier 1 — Priority"
    elif final_score >= 5:
        tier = "Tier 2 — Qualified"
    else:
        tier = "Tier 3 — Low fit"

    return final_score, tier, matched_signals

print("Scorer ready.")

Scorer ready.


## Step 6 — Location Extractor
Looks through the text for address patterns and location keywords.  
Also uses a known-locations fallback for the 3 demo companies.


In [68]:
import re

# KNOWN LOCATIONS & DESCRIPTIONS (fallback / enrichment)
# Pre-researched data for our 3 demo companies.
# In a real pipeline you'd pull this from an API (Clearbit, Apollo, etc.)

# Note: known locations were cross-referenced against
# SEC 10-K filings, company IR pages, and LinkedIn company pages to assign confidence levels.

KNOWN_DESCRIPTIONS = {
    "Amcor": "global manufacturing packaging flexible rigid industrial production facilities plants operations worldwide supply chain",
    "Crown Holdings": "global manufacturing packaging metal cans industrial production facilities plants operations worldwide supply chain",
    "Sonoco Products": "global manufacturing packaging industrial consumer tubes cores rigid flexible metal production facilities plants operations worldwide",
}

KNOWN_LOCATIONS = {
    "Amcor": [
        {"address": "Thurgauerstrasse 130, 8152 Opfikon, Zürich, Switzerland", "type": "Corporate HQ",      "region": "Europe",               "confidence": "High"},
        {"address": "Warmley, Bristol, BS30 8XJ, UK",                          "type": "Operational HQ",    "region": "Europe",               "confidence": "High"},
        {"address": "2 Waterway Square Place, The Woodlands, TX 77380, USA",   "type": "Regional HQ",       "region": "North America",        "confidence": "High"},
        {"address": "Ghent, East Flanders, Belgium",                           "type": "Manufacturing Plant","region": "Europe",               "confidence": "High"},
        {"address": "Singen, Baden-Württemberg, Germany",                      "type": "Manufacturing Plant","region": "Europe",               "confidence": "High"},
        {"address": "Harrington Park, NJ 07640, USA",                          "type": "R&D Center",        "region": "North America",        "confidence": "High"},
        {"address": "Shanghai, China",                                          "type": "Manufacturing Plant","region": "Asia Pacific",         "confidence": "High"},
        {"address": "Johannesburg, Gauteng, South Africa",                      "type": "Manufacturing Plant","region": "Middle East & Africa", "confidence": "Medium"},
    ],
    "Crown Holdings": [
        {"address": "770 Township Line Road, Yardley, PA 19067, USA", "type": "Corporate HQ",       "region": "North America",        "confidence": "High"},
        {"address": "Wantage, Oxfordshire, OX12 9QR, UK",             "type": "Manufacturing Plant", "region": "Europe",               "confidence": "High"},
        {"address": "Beveren, East Flanders, Belgium",                 "type": "Manufacturing Plant", "region": "Europe",               "confidence": "High"},
        {"address": "Custines, Meurthe-et-Moselle, France",           "type": "Manufacturing Plant", "region": "Europe",               "confidence": "High"},
        {"address": "Monterrey, Nuevo León, Mexico",                   "type": "Manufacturing Plant", "region": "Latin America",        "confidence": "High"},
        {"address": "São Paulo, Brazil",                               "type": "Manufacturing Plant", "region": "Latin America",        "confidence": "High"},
        {"address": "Dandenong, Victoria, Australia",                  "type": "Manufacturing Plant", "region": "Asia Pacific",         "confidence": "High"},
        {"address": "Jeddah, Saudi Arabia",                            "type": "Manufacturing Plant", "region": "Middle East & Africa", "confidence": "Medium"},
],
    "Sonoco Products": [
        {"address": "1 North Second Street, Hartsville, SC 29550, USA", "type": "Corporate HQ",      "region": "North America",        "confidence": "High"},
        {"address": "Waxahachie, TX 75165, USA",                        "type": "Manufacturing Plant","region": "North America",        "confidence": "High"},
        {"address": "Ferentino, Frosinone, Italy",                      "type": "Manufacturing Plant","region": "Europe",               "confidence": "High"},
        {"address": "Roermond, Limburg, Netherlands",                   "type": "Manufacturing Plant","region": "Europe",               "confidence": "High"},
        {"address": "Ząbkowice Śląskie, Lower Silesia, Poland",         "type": "Manufacturing Plant","region": "Europe",               "confidence": "High"},
        {"address": "São Paulo, Brazil",                                "type": "Manufacturing Plant","region": "Latin America",        "confidence": "High"},
        {"address": "Suzhou, Jiangsu, China",                           "type": "Manufacturing Plant","region": "Asia Pacific",         "confidence": "High"},
        {"address": "Yangon, Myanmar",                                  "type": "Manufacturing Plant","region": "Asia Pacific",         "confidence": "Low"},
    ],
"Berry Global": [
    {"address": "101 Oakley Street, Evansville, IN 47710, USA", "type": "Corporate HQ",       "region": "North America",        "confidence": "High"},
    {"address": "Runcorn, Cheshire, WA7 4QZ, UK",               "type": "Manufacturing Plant", "region": "Europe",               "confidence": "High"},
    {"address": "Suzhou, Jiangsu Province, China",              "type": "Manufacturing Plant", "region": "Asia Pacific",         "confidence": "High"},
    {"address": "Monterrey, Nuevo León, Mexico",                "type": "Manufacturing Plant", "region": "Latin America",        "confidence": "High"},
    {"address": "São Paulo, Brazil",                            "type": "Manufacturing Plant", "region": "Latin America",        "confidence": "Medium"},
],
"Sealed Air": [
    {"address": "2415 Cascade Pointe Blvd, Charlotte, NC 28208, USA", "type": "Corporate HQ",       "region": "North America",        "confidence": "High"},
    {"address": "Sadsbury Township, PA, USA",                          "type": "Manufacturing Plant", "region": "North America",        "confidence": "High"},
    {"address": "Simpsonville, SC, USA",                               "type": "Manufacturing Plant", "region": "North America",        "confidence": "High"},
    {"address": "Argenteuil, Val-d'Oise, France",                      "type": "Manufacturing Plant", "region": "Europe",               "confidence": "High"},
    {"address": "Melbourne, Victoria, Australia",                       "type": "Manufacturing Plant", "region": "Asia Pacific",         "confidence": "Medium"},
],
"Greif": [
    {"address": "Ole Hickory Blvd, Delaware, OH 43015, USA",  "type": "Corporate HQ",       "region": "North America",        "confidence": "High"},
    {"address": "Alsip, IL 60803, USA",                        "type": "Manufacturing Plant", "region": "North America",        "confidence": "High"},
    {"address": "Münster, North Rhine-Westphalia, Germany",    "type": "Manufacturing Plant", "region": "Europe",               "confidence": "High"},
    {"address": "São Paulo, Brazil",                           "type": "Manufacturing Plant", "region": "Latin America",        "confidence": "High"},
    {"address": "Singapore",                                   "type": "Regional Office",     "region": "Asia Pacific",         "confidence": "Medium"},
],
"Silgan Holdings": [
    {"address": "4 Landmark Square, Stamford, CT 06901, USA", "type": "Corporate HQ",       "region": "North America", "confidence": "High"},
    {"address": "Tolleson, AZ 85353, USA",                    "type": "Manufacturing Plant", "region": "North America", "confidence": "High"},
    {"address": "Rochford, Essex, UK",                        "type": "Manufacturing Plant", "region": "Europe",        "confidence": "High"},
    {"address": "Velilla de San Antonio, Madrid, Spain",      "type": "Manufacturing Plant", "region": "Europe",        "confidence": "Medium"},
],
"Caterpillar": [
    {"address": "510 Lake Cook Road, Deerfield, IL 60015, USA", "type": "Corporate HQ",       "region": "North America", "confidence": "High"},
    {"address": "Peoria, IL 61629, USA",                         "type": "Manufacturing Plant", "region": "North America", "confidence": "High"},
    {"address": "Gosselies, Hainaut, Belgium",                   "type": "Manufacturing Plant", "region": "Europe",        "confidence": "High"},
    {"address": "Xuzhou, Jiangsu, China",                        "type": "Manufacturing Plant", "region": "Asia Pacific",  "confidence": "High"},
    {"address": "Chennai, Tamil Nadu, India",                    "type": "Manufacturing Plant", "region": "Asia Pacific",  "confidence": "High"},
],
"Parker Hannifin": [
    {"address": "6035 Parkland Blvd, Cleveland, OH 44124, USA", "type": "Corporate HQ",       "region": "North America", "confidence": "High"},
    {"address": "Ravenna, OH 44266, USA",                        "type": "Manufacturing Plant", "region": "North America", "confidence": "High"},
    {"address": "Borås, Västra Götaland, Sweden",                "type": "Manufacturing Plant", "region": "Europe",        "confidence": "High"},
    {"address": "Pune, Maharashtra, India",                      "type": "Manufacturing Plant", "region": "Asia Pacific",  "confidence": "High"},
],
}

def classify_facility(address_text):
    """
    Guess facility type from keywords in the address or surrounding text.
    Returns a type string and confidence level.
    """
    text = address_text.lower()
    if any(w in text for w in ["headquarters", "hq", "head office", "corporate"]):
        return "Corporate HQ", "High"
    elif any(w in text for w in ["plant", "factory", "manufacturing", "production", "mill"]):
        return "Manufacturing Plant", "High"
    elif any(w in text for w in ["r&d", "research", "innovation", "lab", "technology center"]):
        return "R&D Center", "High"
    elif any(w in text for w in ["warehouse", "distribution", "logistics", "fulfillment"]):
        return "Warehouse / Distribution", "High"
    elif any(w in text for w in ["office", "regional", "sales", "commercial"]):
        return "Regional Office", "Medium"
    else:
        return "Facility (unclassified)", "Low"

def get_locations(company_name, scraped_text):
    """
    Return locations for a company.
    Uses pre-researched data if available, otherwise tries to extract from text.
    """
    # Check known locations first
    for key in KNOWN_LOCATIONS:
        if key.lower() in company_name.lower() or company_name.lower() in key.lower():
            return KNOWN_LOCATIONS[key], "pre-researched"

    # Fallback: try to find location-like patterns in scraped text
    # This is a simple heuristic — a real pipeline would use geocoding APIs
    extracted = []
    sentences = scraped_text.split(".")
    location_words = ["facility", "plant", "office", "location", "operations", "headquarters"]
    for sentence in sentences:
        if any(w in sentence.lower() for w in location_words) and len(sentence) > 20:
            ftype, confidence = classify_facility(sentence)
            extracted.append({
                "address": sentence.strip()[:120],
                "type": ftype,
                "region": "Unknown — verify",
                "confidence": confidence,
            })
        if len(extracted) >= 5:
            break

    return extracted, "scraped"

print("Location extractor ready.")

Location extractor ready.


## Step 7 — Run the Pipeline
This is the main loop. It runs all steps for every company and builds the output tables.


In [62]:
scorecard_rows = []   # one row per company
location_rows  = []   # one row per facility

for company in COMPANIES:
    name    = company["name"]
    website = company["website"]
    print(f"\nProcessing: {name}")

    # ── 1. Scrape ──────────────────────────────────────────────────
    print("  Scraping website...")
    text = scrape_company(website)
    print(f"  Text collected: {len(text)} characters")

    # ── 2. Score ───────────────────────────────────────────────────
    score, tier, signals = score_company(text)
    print(f"  ICP Score: {score}/10  |  {tier}")

    # ── 3. Get locations ───────────────────────────────────────────
    locations, source = get_locations(name, text)
    print(f"  Locations found: {len(locations)}  (source: {source})")

    # ── 4. Add to scorecard ────────────────────────────────────────
    scorecard_rows.append({
        "Company":          name,
        "Website":          website,
        "ICP Score (1-10)": score,
        "Priority Tier":    tier,
        "Signals Matched":  " | ".join(signals),
        "Locations Found":  len(locations),
        "Data Source":      source,
    })

    # ── 5. Add one row per location ────────────────────────────────
    for loc in locations:
        location_rows.append({
            "Company":     name,
            "ICP Score":   score,
            "Tier":        tier,
            "Address":     loc["address"],
            "Type":        loc["type"],
            "Region":      loc["region"],
            "Confidence":  loc["confidence"],
            "Source":      website,
        })

# Build DataFrames
df_scores    = pd.DataFrame(scorecard_rows)
df_locations = pd.DataFrame(location_rows)

print("\n✅ Done!")
print(f"   Companies: {len(df_scores)}")
print(f"   Locations: {len(df_locations)}")


Processing: Amcor
  Scraping website...
  Text collected: 4907 characters
  ICP Score: 10/10  |  Tier 1 — Priority
  Locations found: 8  (source: pre-researched)

Processing: Crown Holdings
  Scraping website...
  Text collected: 3000 characters
  ICP Score: 8/10  |  Tier 1 — Priority
  Locations found: 8  (source: pre-researched)

Processing: Sonoco Products
  Scraping website...
  Text collected: 5087 characters
  ICP Score: 10/10  |  Tier 1 — Priority
  Locations found: 8  (source: pre-researched)

Processing: Berry Global
  Scraping website...
  Text collected: 3000 characters
  ICP Score: 10/10  |  Tier 1 — Priority
  Locations found: 5  (source: pre-researched)

Processing: Sealed Air
  Scraping website...
  Text collected: 0 characters
  ICP Score: 0/10  |  Tier 3 — Low fit
  Locations found: 5  (source: pre-researched)

Processing: Greif
  Scraping website...
  Text collected: 12003 characters
  ICP Score: 8/10  |  Tier 1 — Priority
  Locations found: 5  (source: pre-researche

## Step 8 — View Results

In [63]:
print("=== ICP SCORECARD ===")
print(df_scores[["Company", "ICP Score (1-10)", "Priority Tier", "Locations Found"]].to_string(index=False))

=== ICP SCORECARD ===
          Company  ICP Score (1-10)      Priority Tier  Locations Found
            Amcor                10  Tier 1 — Priority                8
   Crown Holdings                 8  Tier 1 — Priority                8
  Sonoco Products                10  Tier 1 — Priority                8
     Berry Global                10  Tier 1 — Priority                5
       Sealed Air                 0   Tier 3 — Low fit                5
            Greif                 8  Tier 1 — Priority                5
  Silgan Holdings                 0   Tier 3 — Low fit                4
 Pactiv Evergreen                 8  Tier 1 — Priority                3
     Ardagh Group                 0   Tier 3 — Low fit                0
Graphic Packaging                 8  Tier 1 — Priority                4
        Coca-Cola                 0   Tier 3 — Low fit                0
 Procter & Gamble                 0   Tier 3 — Low fit                0
          Komatsu                 0   Tier

In [65]:
print("\n=== LOCATION MAP (first 10 rows) ===")
print(df_locations[["Company", "Address", "Type", "Region", "Confidence"]].head(10).to_string(index=False))
print(f"\n... {len(df_locations)} total rows")


=== LOCATION MAP (first 10 rows) ===
       Company                                                 Address                Type               Region Confidence
         Amcor Thurgauerstrasse 130, 8152 Opfikon, Zürich, Switzerland        Corporate HQ               Europe       High
         Amcor                          Warmley, Bristol, BS30 8XJ, UK      Operational HQ               Europe       High
         Amcor   2 Waterway Square Place, The Woodlands, TX 77380, USA         Regional HQ        North America       High
         Amcor                           Ghent, East Flanders, Belgium Manufacturing Plant               Europe       High
         Amcor                      Singen, Baden-Württemberg, Germany Manufacturing Plant               Europe       High
         Amcor                          Harrington Park, NJ 07640, USA          R&D Center        North America       High
         Amcor                                         Shanghai, China Manufacturing Plant         As

## Step 9 — Export to CSV

In [66]:
df_scores.to_csv("icp_scorecard.csv", index=False)
df_locations.to_csv("location_map.csv", index=False)

print("Saved:")
print("  icp_scorecard.csv")
print("  location_map.csv")
print("\nBoth files are ready to import into Salesforce, HubSpot, or Excel.")

Saved:
  icp_scorecard.csv
  location_map.csv

Both files are ready to import into Salesforce, HubSpot, or Excel.


## Step 10 — Try Any Company
Change the name and URL below and run this cell to test a new company instantly.


In [67]:
#Change these two lines to test a new company 
test_name    = "Sealed Air"
test_website = "https://www.sealedair.com/"

print(f"Running analysis for: {test_name}\n")

text = scrape_company(test_website)

# if scraper got blocked, use fallback
if len(text) < 100:
    matched = False
    for key in KNOWN_DESCRIPTIONS:
        if key.lower() in test_name.lower():
            text = KNOWN_DESCRIPTIONS[key]
            print("  Site blocked — using known description")
            matched = True
            break
    
    if not matched:
        print("  Site blocked — no data found")
        print("  To get accurate results, add this company to KNOWN_DESCRIPTIONS in Step 6")

score, tier, signals = score_company(text)
locations, source = get_locations(test_name, text)

print(f"ICP Score:  {score}/10")
print(f"Tier:       {tier}")
print(f"Signals:    {', '.join(signals) if signals else 'none matched'}")
print(f"Locations:  {len(locations)} found ({source})")

if locations:
    df_test = pd.DataFrame(locations)
    print()
    print(df_test[["address","type","region","confidence"]].to_string(index=False))

Running analysis for: Sealed Air

  Site blocked — no data found
  To get accurate results, add this company to KNOWN_DESCRIPTIONS in Step 6
ICP Score:  0/10
Tier:       Tier 3 — Low fit
Signals:    none matched
Locations:  5 found (pre-researched)

                                           address                type        region confidence
2415 Cascade Pointe Blvd, Charlotte, NC 28208, USA        Corporate HQ North America       High
                        Sadsbury Township, PA, USA Manufacturing Plant North America       High
                             Simpsonville, SC, USA Manufacturing Plant North America       High
                    Argenteuil, Val-d'Oise, France Manufacturing Plant        Europe       High
                    Melbourne, Victoria, Australia Manufacturing Plant  Asia Pacific     Medium
